# K7: TF-IDF + logistička regresija + numerička i kategorijska kolona (Women's Clothing)
**Tip:** vežba iz tvoje pripreme (PDF `tfidf_lr`). Fokus: `ColumnTransformer` sa tri transformacije (TF-IDF, skaliranje, one-hot).

### Zadatak
U fajlu `Womens Clothing E-Commerce Reviews.csv` su komentari kupaca ženske odeće. Radi se nad uzorkom od 5000 komentara.
- Napraviti `review_content` od `Title` i `Review Text` (prazan naslov ne sme da pokvari spoj).
- Od kolone `Rating` napraviti `sentiment` (1 i 2 negativan, 3 neutralan, 4 i 5 pozitivan). Prikazati udele klasa.
- Za svaku ostalu kolonu obrazložiti uključiti ili ne. Posebno razmotriti `Recommended IND`. Model mora da koristi bar jednu numeričku i bar jednu kategorijsku kolonu pored teksta.
- TF-IDF za tekst, `RobustScaler` za numeričke, one-hot za kategorijske, sve u jednom `ColumnTransformer`-u.
- Prvi model: **logistička regresija**. Metrike, matrica konfuzije, tumačenje (najslabija klasa, macro naspram weighted).
- Drugi model: komentar sa opsezima, pa jedan za TF-IDF i jedan za klasifikator, 5-fold, macro F1.
- Poređenje osnovnog, podešenog i `DummyClassifier`-a.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler, OneHotEncoder, MinMaxScaler, FunctionTransformer
from sklearn import metrics
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, balanced_accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline   # imblearn Pipeline radi i bez SMOTE-a, pa moze da ostane uvek

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier

import matplotlib.pyplot as plt
import seaborn as sb

from pathlib import Path
from IPython.display import display

import spacy

RAND_STATE = 1
DATA_DIR = Path.cwd().parent / 'data'

### Šta je urađeno
- Učitani podaci (prva kolona bez imena se ne koristi), izbačeni komentari bez teksta, uzet uzorak od 5000.
- Napravljena `review_content` i izlazna varijabla `sentiment`.
- Pregledani `Age` i `Positive Feedback Count` (kdeplot), a `Division Name`, `Department Name`, `Class Name` (broj vrednosti i udeo sentimenta po kategoriji).
- `Recommended IND` se ne koristi: kupac ga daje zajedno sa ocenom, pa bi to bilo curenje podataka.
- Predlog ulaza: `review_content`, `Age` (numerička) i `Department Name` (kategorijska, 6 odeljenja). Nekoliko praznih odeljenja popunjeno sa `'Nepoznato'`.
- Podela 80:20 sa `stratify`.

In [ ]:
data = pd.read_csv(DATA_DIR / 'Womens Clothing E-Commerce Reviews.csv', index_col=0)
data.info()

In [ ]:
data = data.dropna(subset=['Review Text'])
data = data.sample(n=5000, random_state=RAND_STATE).reset_index(drop=True)

data['Title'] = data['Title'].fillna('')
data['review_content'] = data['Title'] + '. ' + data['Review Text']
data['Department Name'] = data['Department Name'].fillna('Nepoznato')
print(data.shape)

In [ ]:
def rating_to_sentiment(rating: int):
    if rating < 3:
        return 'negativan'
    return 'neutralan' if rating == 3 else 'pozitivan'

data['sentiment'] = data['Rating'].apply(rating_to_sentiment)
print(data['sentiment'].value_counts(normalize=True))

In [ ]:
for col in ['Age', 'Positive Feedback Count']:
    plt.figure(figsize=(10, 5))
    sb.kdeplot(data=data, x=col, hue='sentiment',
               fill=True, common_norm=False, alpha=0.5, linewidth=2)
    plt.title(f'{col} po sentimentu')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()

print(data[['Division Name', 'Department Name', 'Class Name']].nunique())
print(pd.crosstab(data['Department Name'], data['sentiment'], normalize='index').round(3))
print(pd.crosstab(data['Recommended IND'], data['sentiment'], normalize='index').round(3))

In [ ]:
X = data[['review_content', 'Age', 'Department Name']]
y = data['sentiment']
y_encodings, y_levels = pd.factorize(y)
print(list(y_levels))

X_train, X_test, y_train, y_test = train_test_split(X, y_encodings,
                                                    test_size=0.2,
                                                    random_state=RAND_STATE,
                                                    stratify=y_encodings)
print(X_train.shape, X_test.shape)

---
# ▶ OVDE KREĆEŠ

Sve iznad je gotovo. Od ove ćelije pa nadalje kucaš sam: kod, komentare u markdown ćelijama i tumačenja.

**Tvoji koraci:**

1. spaCy `en_core_web_sm` i `spacy_srediTekst` (zadrži `not`)
2. `ColumnTransformer`: `('text', TfidfVectorizer(...), 'review_content')`, `('scaling', RobustScaler(), ['Age'])`, `('cat', OneHotEncoder(handle_unknown='ignore'), ['Department Name'])`
3. `LogisticRegression(max_iter=2000)` u `Pipeline`, `fit`
4. Metrike, matrica konfuzije, tumačenje
5. Komentar o hiperparametrima, `GridSearchCV` (ime: `data_transform__text__min_df` ako se korak zove `data_transform`)
6. `DummyClassifier(strategy='most_frequent')` i poređenje sva tri

---
## Dati kod
Ovo dobijaš uz zadatak. Pokreni ćeliju kad stigneš do poređenja modela.

In [ ]:
# DATI KOD (dobija se na kolokvijumu): poredjenje modela
def get_model_metrics(y_true: np.ndarray, y_pred: np.ndarray, model_name: str):
    report = classification_report(y_true, y_pred, output_dict=True)
    macro_avg = report['macro avg']
    weighted_avg = report['weighted avg']
    return {
        'Model': model_name,
        'Macro Precision': macro_avg['precision'],
        'Macro Recall': macro_avg['recall'],
        'Macro F1-Score': macro_avg['f1-score'],
        'Weighted Precision': weighted_avg['precision'],
        'Weighted Recall': weighted_avg['recall'],
        'Weighted F1-Score': weighted_avg['f1-score'],
        'Balanced Accuracy': balanced_accuracy_score(y_true, y_pred)
    }